<a href="https://colab.research.google.com/github/faww02/API/blob/main/K-Means.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, precision_recall_fscore_support

# Kriteria ditetapkan SEBELUM menjalankan. Jangan diubah setelah melihat hasil.
SIL_THRESHOLD = 0.5
N_STUDENTS, N_REPEAT = 300, 30   # 300 siswa virtual, diulang 30 kali (seed berbeda)

# ASUMSI peneliti: tiga profil (butuh dukungan rendah, sedang, tinggi) untuk tiga fitur log.
PROFIL = {
    "waktu_detik": {"mean": [40, 70, 110],   "sd": [8, 12, 18]},
    "revisi":      {"mean": [0.3, 1.0, 2.0], "sd": [0.15, 0.3, 0.5]},
    "akurasi":     {"mean": [85, 60, 35],    "sd": [7, 8, 8]},
}

In [ ]:
from google.colab import files
from IPython.display import HTML, display
import html
up = files.upload()                       # pilih Simulator_CT_v1.0.html
isi = list(up.values())[0].decode("utf-8")
display(HTML(f'<iframe srcdoc="{html.escape(isi)}" width="100%" height="650"></iframe>'))

In [ ]:
def make_data(skenario, seed, n=N_STUDENTS):
    """A = tiga kelompok jelas; B = tumpang tindih; C = tanpa struktur (kontrol negatif)."""
    rng = np.random.default_rng(seed)
    asli = rng.integers(0, 3, n)                     # profil asli tiap siswa (0,1,2)
    kolom = {}
    for f, p in PROFIL.items():
        mean, sd = np.array(p["mean"], float), np.array(p["sd"], float)
        if skenario == "B":                          # rapatkan rerata, perbesar sebaran
            mean = mean.mean() + 0.5 * (mean - mean.mean()); sd = sd * 1.3
        if skenario == "C":                          # semua siswa dari satu kelompok yang sama
            mean = np.repeat(mean[1], 3); sd = np.repeat(sd[1], 3)
        kolom[f] = rng.normal(mean[asli], sd[asli])
    df = pd.DataFrame(kolom)
    df["waktu_detik"] = df["waktu_detik"].clip(lower=10)
    df["revisi"] = df["revisi"].clip(lower=0)
    df["akurasi"] = df["akurasi"].clip(0, 100)
    df["profil_asli"] = asli
    return df

In [ ]:
def dispatch(pusat_z, k):
    """Skor kebutuhan dukungan = waktu + revisi - akurasi (nilai terstandar).
    Klaster dengan skor terendah -> Tier 1, tertinggi -> Tier 3."""
    kebutuhan = pusat_z[:, 0] + pusat_z[:, 1] - pusat_z[:, 2]
    peringkat = kebutuhan.argsort().argsort()
    return 1 + (peringkat * 3) // k

def satu_kali(skenario, seed, mode="bebas"):
    df = make_data(skenario, seed)
    X = StandardScaler().fit_transform(df[["waktu_detik", "revisi", "akurasi"]])
    terbaik = None
    for k in ([3] if mode == "k3" else range(2, 6)):          # "k3" = k dikunci 3 (sesuai 3 tier)
        km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X)
        s = silhouette_score(X, km.labels_)
        if terbaik is None or s > terbaik[0]:
            terbaik = (s, k, km)
    s, k, km = terbaik
    tier_pred = dispatch(km.cluster_centers_, k)[km.labels_]
    hasil = {"mode": mode, "silhouette": s, "k_terpilih": k}
    if skenario != "C":                                        # pada C tidak ada struktur asli
        tier_asli = df["profil_asli"].to_numpy() + 1
        hasil["ARI"] = adjusted_rand_score(df["profil_asli"], km.labels_)
        p, r, _, _ = precision_recall_fscore_support(tier_asli, tier_pred, average="macro", zero_division=0)
        hasil["presisi_makro"], hasil["recall_makro"] = p, r
    return hasil

In [ ]:
baris = []
for mode in ["bebas", "k3"]:
    for sk in ["A", "B", "C"]:
        for seed in range(N_REPEAT):
            baris.append({"skenario": sk, **satu_kali(sk, seed, mode)})
hasil = pd.DataFrame(baris)
tabel = hasil.groupby(["mode", "skenario"]).agg(["mean", "std"]).round(3)
print(tabel.T)
print("Silhouette ≥", SIL_THRESHOLD, "?")
print(hasil.groupby(["mode", "skenario"])["silhouette"].apply(lambda s: (s >= SIL_THRESHOLD).mean()).round(2))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, sk in zip(ax, ["A", "B", "C"]):
    d = make_data(sk, 0)
    a.scatter(d["waktu_detik"], d["akurasi"], c=d["profil_asli"], s=10)
    a.set_title(f"Skenario {sk}"); a.set_xlabel("waktu (detik)"); a.set_ylabel("akurasi")
plt.tight_layout(); plt.show()